<a href="https://colab.research.google.com/github/fyremael/TROPICA/blob/main/notebooks/03_researcher_showcase.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TROPICA Researcher Showcase

Goal: compile bounded JSON/tool-call specs into real tokenizer-ID masks, drive the decoder with offline logit providers, inspect trace events, and exercise fail-closed controls.

In [ ]:
import pathlib
import subprocess
import sys

repo_root = pathlib.Path.cwd()
if (repo_root / "pyproject.toml").exists():
    install_cmd = [sys.executable, "-m", "pip", "install", "-q", "-e", ".[real-tokenizers,dev]"]
else:
    install_cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "control-delta-support-decoding[real-tokenizers,dev] @ git+https://github.com/fyremael/TROPICA.git",
    ]
subprocess.check_call(install_cmd)


In [ ]:
from cdsd import TiktokenAdapter, ToolCallSpec, StructuredOutputCompiler

specs = [
    ToolCallSpec(
        "search",
        {
            "type": "object",
            "required": ["query", "limit"],
            "properties": {
                "query": {"type": "string", "enum": ["alpha", "beta", "unicode: lambda"]},
                "limit": {"type": "integer", "enum": [1, 2, 3]},
            },
            "additionalProperties": False,
        },
    ),
    ToolCallSpec(
        "route",
        {
            "type": "object",
            "required": ["lane", "urgent"],
            "properties": {
                "lane": {"type": "string", "enum": ["planner", "tokenizer", "report"]},
                "urgent": {"type": "boolean", "enum": [True, False]},
            },
            "additionalProperties": False,
        },
    ),
]

adapter = TiktokenAdapter("cl100k_base")
compiler = StructuredOutputCompiler(adapter, specs)
print("adapter:", adapter.name)
print("canonical outputs:", len(compiler.outputs))
compiler.outputs[:3]


In [ ]:
from cdsd import HostileLogitProvider, StructuredOutputDecoder

decoder = StructuredOutputDecoder(compiler)
hostile = HostileLogitProvider(illegal_token_ids=(0, 1, 2, 999_999_001))
result = decoder.decode(hostile, max_steps=256)
print("accepted:", result.accepted)
print("value:", result.value)
print("parsed:", result.parsed)
print("steps:", result.steps)
assert result.accepted and result.parsed is not None
assert any(event.top_illegal_score is not None and event.top_illegal_score > event.selected_score for event in result.events)


In [ ]:
from IPython.display import HTML, display
import html

rows = [
    "<tr><th>step</th><th>allowed</th><th>selected</th><th>score</th><th>top illegal</th><th>accepted</th></tr>"
]
for event in result.events[:20]:
    top_illegal = "" if event.top_illegal_token_id is None else f"{event.top_illegal_token_id} @ {event.top_illegal_score:.1f}"
    rows.append(
        "<tr>"
        f"<td>{event.step}</td>"
        f"<td>{event.allowed_count}</td>"
        f"<td>{event.selected_token_id}</td>"
        f"<td>{event.selected_score:.1f}</td>"
        f"<td>{html.escape(top_illegal)}</td>"
        f"<td>{event.accepted}</td>"
        "</tr>"
    )
display(HTML("<table>" + "".join(rows) + "</table>"))


In [ ]:
from cdsd import ScriptedLogitProvider

target = compiler.outputs[-1]
scripted = ScriptedLogitProvider(adapter.encode(target), illegal_token_ids=(0, 1, 2))
exact = decoder.decode(scripted, max_steps=256)
print(exact.value)
assert exact.value == target
assert exact.parsed is not None


In [ ]:
from cdsd import CallableLogitProvider

calls = []
def choose_smallest_legal(emitted, allowed):
    calls.append((emitted, set(allowed)))
    chosen = min(allowed)
    return {token_id: 0.0 for token_id in allowed} | {chosen: 1.0, 999_999_002: 1_000_000.0}

callable_result = decoder.decode(CallableLogitProvider(choose_smallest_legal), max_steps=256)
print("accepted:", callable_result.accepted)
print("callback calls:", len(calls))
print("value:", callable_result.value)
assert callable_result.accepted
assert len(calls) == callable_result.steps


In [ ]:
from cdsd import StructuredOutputDecodeError, TokenizerPrefixError

try:
    decoder.decode(HostileLogitProvider(), max_steps=1)
    raise AssertionError("expected max-step exhaustion")
except StructuredOutputDecodeError as exc:
    print("max-step exhaustion raised:", type(exc).__name__)

try:
    compiler.update(compiler.initial_state(), 999_999_003)
    raise AssertionError("expected illegal transition")
except TokenizerPrefixError as exc:
    print("illegal transition raised:", type(exc).__name__)


Interpretation: this is the model-facing shape of TROPICA. A provider supplies scores, the compiler supplies real tokenizer-ID support, and the decoder chooses only legal token IDs while recording enough trace evidence to audit each step. To plug in a local model later, implement `LogitProvider.next_logits(emitted_token_ids, allowed_token_ids)` and return a mapping from token IDs to scores.